# Penalty prediction, take 3: attack/defense (Dixon-Coles-shaped) structure

`penalty_prediction.ipynb` (match-level XGBoost on rolling stats) and `penalty_season_rate.ipynb`
(season-level Poisson on prior-season team quality, plus in-season Bayesian updating) both
concluded **no reliable edge over a flat home/away/league baseline** — for two different,
diagnosable reasons: match-level rolling stats are too noisy to beat the free baseline, and a
single prior season's PPG/GD is too weak a forecast of *this* season's penalty rate because team
quality doesn't persist cleanly across the close-season.

Both of those notebooks only ever used **own-team** features at the season level. Neither ever
built an explicit two-sided structure — the goal-scoring equivalent would be Dixon-Coles'
attack(i) x defense(j) x home-advantage. This notebook asks five questions aimed at that gap,
using two data sources neither previous notebook touched:

- `match_events` (WhoScored, per-event x/y — Premier League only, 2021-2022 onward): lets us
  compute *zone-specific* stats FotMob's match-level aggregates can't — touches/dribbles won
  inside the box, fouls committed inside a team's own box specifically (not just fouls-per-game).
- `xt` / `xt_actions` (this project's self-fit gross_xT, Premier League only): a cumulative
  progressive-value metric, distinct from shot-quality-weighted xG.

**Scope note:** both of these are Premier League only (WhoScored's coverage), so questions 2-5
below are PL-only. Question 1's primary test uses xG (available for all three leagues) with a
PL-only confirmatory check.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import chi2
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

DB_PATH = '../../infra/data/db/fotmob.db'
conn = sqlite3.connect(DB_PATH)

# WhoScored -> FotMob team name normalisation, extending the project's existing convention
# (src/data_utils.py:WS_TO_FM_NAMES) with two more teams that appear in this dataset's window.
WS_TO_FM_NAMES = {
    'Manchester City': 'Man City', 'Manchester United': 'Man United',
    'Nottingham Forest': 'Nottm Forest', 'Sheffield United': 'Sheff Utd', 'Luton': 'Luton Town',
}
tid = pd.read_sql_query('SELECT team_id, team_name FROM team_id_mapping', conn)
name2id = dict(zip(tid.team_name, tid.team_id))

matches = pd.read_sql_query(
    "SELECT match_id, home_team, away_team, season, match_date, home_goals, away_goals "
    "FROM matches WHERE league_id='Premier_League'", conn)
matches['match_date'] = pd.to_datetime(matches['match_date'])
matches = matches[matches.season != '2026-2027']  # in progress

pens = pd.read_sql_query("SELECT match_id, home_pens, away_pens FROM penalties", conn)
matches = matches.merge(pens, on='match_id', how='left')
matches[['home_pens', 'away_pens']] = matches[['home_pens', 'away_pens']].fillna(0)

ms = pd.read_sql_query(
    "SELECT match_id, home_expected_goals, away_expected_goals FROM match_stats", conn)
ms['home_expected_goals'] = pd.to_numeric(ms['home_expected_goals'], errors='coerce')
ms['away_expected_goals'] = pd.to_numeric(ms['away_expected_goals'], errors='coerce')
matches = matches.merge(ms, on='match_id', how='left')

print(matches.shape)
matches.head(3)

(2280, 11)


,match_id,home_team,away_team,season,match_date,home_goals,away_goals,home_pens,away_pens,home_expected_goals,away_expected_goals
0,3610280,9850,8654,2021-2022,2022-05-08,0,4,0,1,0.78,3.13
1,3610003,8197,10260,2021-2022,2021-10-16,4,2,0,0,2.54,1.16
2,3610146,9850,8668,2021-2022,2022-01-15,2,1,0,0,1.00,0.61


## Verifying the coordinate convention before trusting any zone feature

`match_events.x/y` run 0-100. A code comment in `infra/data/collectors/whoscored/xt_model.py`
states x=100 is always the goal the team-in-possession is attacking — i.e. coordinates are
normalised **per team**, not to a fixed pitch orientation. Before building zone-filtered
features on that assumption, verify it against ground truth: sum `penaltyWon` events per
team-match and check they match `penalties.home_pens`/`away_pens` exactly. This also validates
the WhoScored-matchId <-> FotMob-match_id bridge (joined via home/away team name + season, since
the two scrapers use unrelated match ID spaces).

In [2]:
me_meta = pd.read_sql_query(
    "SELECT DISTINCT matchId, homeTeam, awayTeam, season FROM match_events", conn)
me_meta['home_fm'] = me_meta.homeTeam.map(lambda n: WS_TO_FM_NAMES.get(n, n))
me_meta['away_fm'] = me_meta.awayTeam.map(lambda n: WS_TO_FM_NAMES.get(n, n))
me_meta['home_id'] = me_meta.home_fm.map(name2id)
me_meta['away_id'] = me_meta.away_fm.map(name2id)
assert me_meta.home_id.notna().all() and me_meta.away_id.notna().all(), 'unmapped team name'

bridge = me_meta.merge(
    matches[['match_id', 'home_team', 'away_team', 'season']],
    left_on=['home_id', 'away_id', 'season'], right_on=['home_team', 'away_team', 'season'], how='inner')
assert bridge.groupby('matchId').match_id.nunique().max() == 1, 'ambiguous matchId bridge'
matchid_map = dict(zip(bridge.matchId, bridge.match_id))
print(f'{len(bridge)}/{len(me_meta)} WhoScored matches bridged to a unique FotMob match_id')

pw = pd.read_sql_query(
    "SELECT matchId, h_a, SUM(penaltyWon) as pw FROM match_events GROUP BY matchId, h_a", conn)
pw['match_id'] = pw.matchId.map(matchid_map)
pw = pw.dropna(subset=['match_id'])
pw_wide = pw.pivot_table(index='match_id', columns='h_a', values='pw', fill_value=0).rename(
    columns={'h': 'ws_home_pw', 'a': 'ws_away_pw'})
check = pw_wide.merge(matches[['match_id', 'home_pens', 'away_pens']], on='match_id', how='left')
print('home penaltyWon matches home_pens:', (check.ws_home_pw == check.home_pens).mean())
print('away penaltyWon matches away_pens:', (check.ws_away_pw == check.away_pens).mean())

1837/1877 WhoScored matches bridged to a unique FotMob match_id


home penaltyWon matches home_pens: 1.0
away penaltyWon matches away_pens: 0.9994556341861731


Matches on 100% of home rows and 99.95% of away rows (1 mismatch out of ~1,880 — negligible,
not chased further). The bridge and the per-team coordinate convention both check out. Spot-checking
one penalty directly confirms the row-level semantics too: the fouled/attacking team's row has
`foulGiven=1, penaltyWon=1` at `x≈89` (deep in their attacking third), and the same instant's
defending team's row has `penaltyConceded=1` at `x≈11` — i.e. `foulCommitted` is recorded on the
*defending* team's own row, in *that team's own* attacking-direction frame, so `x≈11` for the
defender is the same physical spot on the pitch as `x≈89` for the attacker. That means:
- **"opponent's box" for an attacking-team's own event** = `x >= 83, 21 <= y <= 79`
- **"own box" for a defending-team's own event** (e.g. a foul committed while defending deep) = `x <= 17, 21 <= y <= 79`

(17/100 approximates a real penalty box's ~16.5m depth on a ~105m pitch; y-band approximates the box's width.)

In [3]:
me = pd.read_sql_query(
    "SELECT matchId, teamId, h_a, homeTeam, awayTeam, season, x, y, isTouch, dribbleWon, foulCommitted "
    "FROM match_events", conn)
me['home_fm'] = me.homeTeam.map(lambda n: WS_TO_FM_NAMES.get(n, n))
me['away_fm'] = me.awayTeam.map(lambda n: WS_TO_FM_NAMES.get(n, n))
me['home_id'] = me.home_fm.map(name2id)
me['away_id'] = me.away_fm.map(name2id)
me['match_id'] = me.matchId.map(matchid_map)
me = me.dropna(subset=['match_id'])

in_opp_box = (me.x >= 83) & (me.y.between(21, 79))
in_own_box = (me.x <= 17) & (me.y.between(21, 79))
me['opp_box_touch'] = np.where(in_opp_box & (me.isTouch == 1), 1, 0)
me['opp_box_dribble_won'] = np.where(in_opp_box & (me.dribbleWon == 1), 1, 0)
me['own_box_foul'] = np.where(in_own_box & (me.foulCommitted == 1), 1, 0)

zone = me.groupby(['match_id', 'teamId', 'h_a']).agg(
    opp_box_touch=('opp_box_touch', 'sum'),
    opp_box_dribble_won=('opp_box_dribble_won', 'sum'),
    own_box_foul=('own_box_foul', 'sum'),
).reset_index()
zone_home = zone[zone.h_a == 'h'].set_index('match_id')[['opp_box_touch', 'opp_box_dribble_won', 'own_box_foul']]
zone_home.columns = [f'home_{c}' for c in zone_home.columns]
zone_away = zone[zone.h_a == 'a'].set_index('match_id')[['opp_box_touch', 'opp_box_dribble_won', 'own_box_foul']]
zone_away.columns = [f'away_{c}' for c in zone_away.columns]
matches = matches.merge(zone_home, on='match_id', how='left').merge(zone_away, on='match_id', how='left')

xt = pd.read_sql_query("SELECT matchId, team, season, gross_xT FROM xt", conn)
xt['team_fm'] = xt.team.map(lambda n: WS_TO_FM_NAMES.get(n, n))
xt['team_id'] = xt.team_fm.map(name2id)

print(f'{matches.home_opp_box_touch.notna().sum()}/{len(matches)} PL matches have zone features '
      '(gap = 2020-2021, before WhoScored coverage here)')

1837/2280 PL matches have zone features (gap = 2020-2021, before WhoScored coverage here)


## Season-team panel

One row per team-season, Premier League 2021-2022 through 2025-2026 (the seasons with full
`match_events`/`xt` coverage). `games >= 25` filters incomplete seasons.

In [4]:
def build_season_team(matches, me_zone_df):
    home = matches[['season', 'match_id', 'home_team', 'away_team', 'home_pens',
                     'home_expected_goals', 'away_expected_goals',
                     'home_opp_box_touch', 'home_opp_box_dribble_won', 'home_own_box_foul']].rename(columns={
        'home_team': 'team_id', 'away_team': 'opp_id', 'home_pens': 'pens_for',
        'home_expected_goals': 'xg_for', 'away_expected_goals': 'xg_against',
        'home_opp_box_touch': 'opp_box_touch', 'home_opp_box_dribble_won': 'opp_box_dribble_won',
        'home_own_box_foul': 'own_box_foul'})
    away = matches[['season', 'match_id', 'away_team', 'home_team', 'away_pens',
                     'away_expected_goals', 'home_expected_goals',
                     'away_opp_box_touch', 'away_opp_box_dribble_won', 'away_own_box_foul']].rename(columns={
        'away_team': 'team_id', 'home_team': 'opp_id', 'away_pens': 'pens_for',
        'away_expected_goals': 'xg_for', 'home_expected_goals': 'xg_against',
        'away_opp_box_touch': 'opp_box_touch', 'away_opp_box_dribble_won': 'opp_box_dribble_won',
        'away_own_box_foul': 'own_box_foul'})
    panel = pd.concat([home, away], ignore_index=True)

    against = panel[['season', 'match_id', 'team_id', 'opp_id', 'pens_for']].rename(
        columns={'team_id': 't', 'opp_id': 'team_id', 'pens_for': 'pens_against'})
    pa = against.groupby(['season', 'team_id'])['pens_against'].sum().reset_index()

    st = panel.groupby(['season', 'team_id']).agg(
        games=('pens_for', 'size'), pens_for=('pens_for', 'sum'),
        xg_for=('xg_for', 'sum'), xg_against=('xg_against', 'sum'),
        opp_box_touch=('opp_box_touch', 'sum'), opp_box_dribble_won=('opp_box_dribble_won', 'sum'),
        own_box_foul=('own_box_foul', 'sum'),
    ).reset_index()
    st = st.merge(pa, on=['season', 'team_id'], how='left')
    st = st[st.games >= 25].reset_index(drop=True)
    return st

st = build_season_team(matches[matches.season != '2020-2021'], me)
xt_season = xt.dropna(subset=['team_id']).groupby(['season', 'team_id'])['gross_xT'].sum().reset_index()
st = st.merge(xt_season, on=['season', 'team_id'], how='left')

for num, den in [('pens_for', 'games'), ('pens_against', 'games'), ('xg_for', 'games'),
                  ('xg_against', 'games'), ('gross_xT', 'games'), ('opp_box_touch', 'games'),
                  ('opp_box_dribble_won', 'games'), ('own_box_foul', 'games')]:
    st[f'{num}_pg'] = st[num] / st[den]

seasons = sorted(st.season.unique())
season_idx = {s: i for i, s in enumerate(seasons)}
st['sidx'] = st.season.map(season_idx)
print(st.shape, seasons)
st.head()

(100, 20) ['2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']


,season,team_id,games,pens_for,xg_for,xg_against,opp_box_touch,opp_box_dribble_won,own_box_foul,pens_against,gross_xT,pens_for_pg,pens_against_pg,xg_for_pg,xg_against_pg,gross_xT_pg,opp_box_touch_pg,opp_box_dribble_won_pg,own_box_foul_pg,sidx
0,2021-2022,8191,38,2,40.93,58.57,656.0,13.0,5.0,5,69.975321,0.052632,0.131579,1.077105,1.541316,1.841456,17.263158,0.342105,0.131579,0
1,2021-2022,8197,38,2,48.26,61.00,742.0,17.0,8.0,9,69.646055,0.052632,0.236842,1.270000,1.605263,1.832791,19.526316,0.447368,0.210526,0
2,2021-2022,8455,38,9,65.23,34.56,1047.0,21.0,5.0,6,101.746448,0.236842,0.157895,1.716579,0.909474,2.677538,27.552632,0.552632,0.131579,0
3,2021-2022,8456,38,9,90.52,24.75,1507.0,47.0,1.0,1,125.239559,0.236842,0.026316,2.382105,0.651316,3.295778,39.657895,1.236842,0.026316,0
4,2021-2022,8463,38,5,45.24,69.43,793.0,23.0,5.0,5,70.060237,0.131579,0.131579,1.190526,1.827105,1.843690,20.868421,0.605263,0.131579,0


## Q1 — Residual test: does xT explain penalty rate beyond raw attack strength?

The observation from chat that kicked this off: a competitor model apparently shows two teams
with *equal attacking strength* drawing different penalty rates. Operationalise that directly —
regress season pens-for on an attack-strength proxy, then check whether the leftover residual
correlates with `gross_xT`.

Two attack-strength proxies, at different levels of "quality-adjustment":
- **season-total xG** (all 3 leagues available, but run PL-only here for consistency with the
  rest of this notebook) — a shot-volume/quality measure, still fairly close to raw output.
- **this project's own Bayesian `att_str_raw` posterior mean** (PL only, and only one season
  of trace currently saved: 2025-2026) — a *proper* quality-adjusted attacking-strength
  estimate (nets out opponent strength, home advantage, etc. within the hierarchical model).
  Thin sample (n=20, one season), so treat as confirmatory, not primary evidence.

In [5]:
m1 = smf.glm('pens_for ~ xg_for', data=st, offset=np.log(st.games), family=sm.families.Poisson()).fit()
print(m1.summary().tables[1])
st['resid_pens_vs_xg'] = st.pens_for - m1.predict(st)

r_resid, p_resid = stats.pearsonr(st.resid_pens_vs_xg, st.gross_xT)
r_raw_xg, p_raw_xg = stats.pearsonr(st.pens_for_pg, st.xg_for_pg)
r_raw_xt, p_raw_xt = stats.pearsonr(st.pens_for_pg, st.gross_xT_pg)
print(f'\nraw corr(pens_for_pg, xg_for_pg)       r={r_raw_xg:.3f}  p={p_raw_xg:.4f}')
print(f'raw corr(pens_for_pg, gross_xT_pg)      r={r_raw_xt:.3f}  p={p_raw_xt:.4f}')
print(f'corr(residual-after-xG, gross_xT)       r={r_resid:.3f}  p={p_resid:.4f}')

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.2809      0.188    -17.474      0.000      -3.649      -2.913
xg_for         0.0214      0.003      6.983      0.000       0.015       0.027

raw corr(pens_for_pg, xg_for_pg)       r=0.586  p=0.0000
raw corr(pens_for_pg, gross_xT_pg)      r=0.416  p=0.0000
corr(residual-after-xG, gross_xT)       r=0.406  p=0.0000


In [6]:
att = pd.read_csv(
    '../premier_league/model_traces/v1/summary_Premier_League_2025-2026.csv'.replace(
        '../premier_league', '../../algo/models/team_strength/non_penalty_bayes/premier_league'))
att = att[att.iloc[:, 0].str.startswith('att_str_raw')].copy()
att['team_name'] = att.iloc[:, 0].str.extract(r'\[(.+)\]')
att = att.merge(tid, on='team_name', how='left')

st2526 = st[st.season == '2025-2026'].merge(att[['team_id', 'mean']], on='team_id', how='left') \
    .rename(columns={'mean': 'att_str_raw'}).dropna(subset=['att_str_raw'])
r_att, p_att = stats.pearsonr(st2526.pens_for_pg, st2526.att_str_raw)
print(f'2025-2026 cross-section (n={len(st2526)}): corr(pens_for_pg, att_str_raw) r={r_att:.3f} p={p_att:.4f}')
print(f'(for comparison, same season) corr(pens_for_pg, xg_for_pg) '
      f'r={stats.pearsonr(st2526.pens_for_pg, st2526.xg_for_pg)[0]:.3f}, '
      f'corr(pens_for_pg, gross_xT_pg) r={stats.pearsonr(st2526.pens_for_pg, st2526.gross_xT_pg)[0]:.3f}')

2025-2026 cross-section (n=20): corr(pens_for_pg, att_str_raw) r=0.109 p=0.6472
(for comparison, same season) corr(pens_for_pg, xg_for_pg) r=0.457, corr(pens_for_pg, gross_xT_pg) r=0.012


### Reading the Q1 result

Two things point in different directions, and both matter:

- **xG is the single strongest predictor of penalty rate of anything tried in either this or the
  prior two notebooks** (r=0.586 across 100 team-seasons) — clearly stronger than gross_xT alone
  (r=0.416). So the naive version of "xT beats attack strength" is *not* what the data shows.
- But the **residual test says gross_xT is not redundant with xG** — after netting out what xG
  already explains, the leftover in penalty rate still correlates with gross_xT at r=0.41
  (p<0.0001). xT is adding real signal on top of xG, not just a noisier version of the same
  thing.
- The **quality-adjusted `att_str_raw` posterior shows ~zero correlation with penalty rate**
  (r=0.11, p=0.65, n=20 — thin, one-season sample, treat cautiously) even though same-season xG
  and gross_xT both showed strong correlations. That's the sharpest version of the chat
  observation: a team's *net attacking quality* (adjusted for opponent strength, which is what
  the Bayesian rating targets) looks unrelated to penalty rate, while *raw danger-zone volume*
  (xG, xT) is what actually tracks it. Penalty rate looks like a function of how much time a team
  spends generating shots/actions near the box, not how *good* it is at converting that time into
  goals net of who it played.

## Q2 — Mechanism check: which proxy is tightest for pens-for?

xG is an aggregate, shot-outcome-weighted number. `match_events` lets us go more granular:
touches and dribbles won specifically inside the opponent's box — the literal, physical
precursor to drawing a penalty, with no shot-quality weighting mixed in. Does the more granular,
more "mechanistic" feature beat the aggregate?

In [7]:
for col in ['xg_for_pg', 'gross_xT_pg', 'opp_box_touch_pg', 'opp_box_dribble_won_pg']:
    r, p = stats.pearsonr(st[col], st.pens_for_pg)
    print(f'{col:22s} r={r:.3f}  p={p:.4f}')

xg_for_pg              r=0.586  p=0.0000
gross_xT_pg            r=0.416  p=0.0000
opp_box_touch_pg       r=0.392  p=0.0001
opp_box_dribble_won_pg r=0.252  p=0.0113


### Reading the Q2 result

xG stays on top (r=0.586); the zone-filtered `match_events` features (box touches r=0.39, box
dribbles won r=0.25) don't beat it, and neither does gross_xT (r=0.42). The more literal,
un-weighted "how much do you physically occupy the box" signal is *not* a tighter proxy than the
aggregate shot-quality number. Read together with Q1, the honest picture: xG is doing double duty
as both a volume-of-danger signal and a quality signal, and that combination — not either
component split out on its own — is what tracks penalty rate best. gross_xT adds a smaller,
genuinely independent second signal on top (per the Q1 residual test), but doesn't replace xG as
the primary covariate.

## Q3 — Opponent concession as its own trait

Neither prior notebook ever built an opponent-side season-level feature — `penalty_season_rate.ipynb`
only ever regressed a team's *own* prior PPG/GD/pens-for on its *own* future pens-for. Test the
concession side directly: does an opponent's **own-box foul rate** (fouls committed specifically
in their own box, from `match_events`) explain pens-conceded better than their general defensive
quality (xG conceded)?

In [8]:
for col in ['xg_against_pg', 'own_box_foul_pg']:
    r, p = stats.pearsonr(st[col], st.pens_against_pg)
    print(f'{col:18s} r={r:.3f}  p={p:.4f}')

m3a = smf.glm('pens_against ~ xg_against', data=st, offset=np.log(st.games), family=sm.families.Poisson()).fit()
m3b = smf.glm('pens_against ~ xg_against + own_box_foul_pg', data=st, offset=np.log(st.games),
              family=sm.families.Poisson()).fit()
print(f'\ndeviance, xg_against only:            {m3a.deviance:.1f}  AIC {m3a.aic:.1f}')
print(f'deviance, xg_against + own_box_foul:  {m3b.deviance:.1f}  AIC {m3b.aic:.1f}')
lr_p = 1 - chi2.cdf(m3a.deviance - m3b.deviance, 1)
print(f'likelihood-ratio test p-value: {lr_p:.2e}')
print(m3b.summary().tables[1])

xg_against_pg      r=0.567  p=0.0000
own_box_foul_pg    r=0.964  p=0.0000

deviance, xg_against only:            93.4  AIC 425.7
deviance, xg_against + own_box_foul:  22.3  AIC 356.6
likelihood-ratio test p-value: 0.00e+00
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -3.2442      0.237    -13.664      0.000      -3.710      -2.779
xg_against          0.0032      0.005      0.669      0.503      -0.006       0.013
own_box_foul_pg     7.2520      0.861      8.425      0.000       5.565       8.939


Same-season, `own_box_foul_pg` correlates with `pens_against_pg` at **r=0.964** — an almost
mechanical relationship — and once it's in the model, `xg_against`'s coefficient stops being
significant at all (p=0.50). That looks like a dramatic win for the "opponent has a discipline
trait, not just a defensive-quality trait" hypothesis. **But this needs a persistence check
before it's trusted as a real, forecastable team trait** — a same-season correlation this strong
is closer to definitional overlap (a penalty *is*, almost by construction, a subset of "fouls
committed in the box") than to an independent predictive signal. The real test: does *last*
season's box-foul rate predict *this* season's penalties conceded, the way `penalty_season_rate.ipynb`
tested for PPG/GD?

In [9]:
prior = st[['team_id', 'sidx', 'xg_against_pg', 'own_box_foul_pg', 'pens_against_pg',
            'xg_for_pg', 'gross_xT_pg', 'pens_for_pg']].copy()
prior['sidx'] = prior['sidx'] + 1
prior = prior.rename(columns={c: f'prior_{c}' for c in prior.columns if c not in ('team_id', 'sidx')})
merged = st.merge(prior, on=['team_id', 'sidx'], how='inner')
print(f'n consecutive team-season pairs: {len(merged)}\n')

print('does PRIOR season predict THIS season pens_against_pg?')
for col in ['prior_xg_against_pg', 'prior_own_box_foul_pg', 'prior_pens_against_pg']:
    r, p = stats.pearsonr(merged[col], merged.pens_against_pg)
    print(f'  {col:25s} r={r:.3f} p={p:.4f}')

print('\ndoes PRIOR season predict THIS season pens_for_pg?')
for col in ['prior_xg_for_pg', 'prior_gross_xT_pg', 'prior_pens_for_pg']:
    r, p = stats.pearsonr(merged[col], merged.pens_for_pg)
    print(f'  {col:25s} r={r:.3f} p={p:.4f}')

print('\nis the raw metric itself persistent season-to-season (autocorrelation)?')
for a, b in [('prior_own_box_foul_pg', 'own_box_foul_pg'), ('prior_xg_against_pg', 'xg_against_pg'),
             ('prior_xg_for_pg', 'xg_for_pg'), ('prior_gross_xT_pg', 'gross_xT_pg')]:
    r, p = stats.pearsonr(merged[a], merged[b])
    print(f'  {a:25s} -> {b:15s} r={r:.3f} p={p:.4f}')

n consecutive team-season pairs: 68

does PRIOR season predict THIS season pens_against_pg?
  prior_xg_against_pg       r=0.312 p=0.0096
  prior_own_box_foul_pg     r=0.130 p=0.2891
  prior_pens_against_pg     r=0.154 p=0.2094

does PRIOR season predict THIS season pens_for_pg?
  prior_xg_for_pg           r=0.363 p=0.0024
  prior_gross_xT_pg         r=0.361 p=0.0025
  prior_pens_for_pg         r=0.076 p=0.5380

is the raw metric itself persistent season-to-season (autocorrelation)?
  prior_own_box_foul_pg     -> own_box_foul_pg r=0.155 p=0.2082
  prior_xg_against_pg       -> xg_against_pg   r=0.549 p=0.0000
  prior_xg_for_pg           -> xg_for_pg       r=0.645 p=0.0000
  prior_gross_xT_pg         -> gross_xT_pg     r=0.821 p=0.0000


### Reading the Q3 result — the persistence check overturns the headline number

The r=0.964 same-season correlation **does not survive to a genuine forecast**:
`prior_own_box_foul_pg` predicts *next* season's pens-conceded at only r=0.13 (p=0.29, not
significant) — worse than `prior_xg_against_pg` (r=0.31, p=0.01). And `own_box_foul_pg` isn't
even persistent as a trait in its own right — its season-to-season autocorrelation is r=0.16
(not significant), versus r=0.55-0.82 for xG/xT. **A team's own-box foul rate bounces around too
much year to year to be a real, forecastable "dirty defending" trait** — the eye-catching
same-season number is mostly definitional overlap (fouls in the box and penalties conceded are
drawn from the same event category), not a stable style signal.

The more interesting, more honest finding: **opponent concession is real, and it IS distinct
from a team's own attack** (Q1/Q2 established the attack side runs through xG/xT) — but the thing
that actually persists on the concession side is the same kind of signal as the attack side:
general "how much danger do they concede" (xG against), not a special disciplinary trait. That's
enough to build the two-sided structure Q4 is after, just with duller, sturdier covariates than
the exciting-looking box-foul number.

## Q4 — Two-parameter multiplicative structure (Dixon-Coles shaped)

Given Q1-Q3: use `xg_for_pg` for the attack axis and `xg_against_pg` for the concession axis —
both persist season-to-season at a level nothing else tested does. Build team-level attack/defense
*effects* with **empirical-Bayes shrinkage pooled across all prior seasons with exponential
decay**, not a single prior-season point estimate (that specific choice — one lagged season, no
pooling — is what left `penalty_season_rate.ipynb`'s season model "tied with a flat mean").

One deliberate deviation from the project's usual `decay_rate=0.0077`/day convention (used for
the referee feature): that rate has a ~90-day half-life, which would put *two seasons ago* at a
weight of `exp(-0.0077*730) ≈ 0.004` — collapsing multi-season pooling straight back to a
single-prior-season point estimate, the exact thing being avoided here. Season-grain persistence
needs a much slower decay, so instead of transplanting the day-level constant, `rho` (per-season
decay) and `k` (shrinkage pseudo-count) are grid-searched directly against held-out deviance —
same idiom as the `WINDOWS`/`K_VALUES` sweeps in the two prior notebooks.

In [10]:
FOLDS = [
    (['2021-2022'], '2022-2023'),
    (['2021-2022', '2022-2023'], '2023-2024'),
    (['2021-2022', '2022-2023', '2023-2024'], '2024-2025'),
    (['2021-2022', '2022-2023', '2023-2024', '2024-2025'], '2025-2026'),
]

def shrunk_effect(team_hist, league_mean_by_season, target_sidx, rho, k):
    # team_hist: rows (sidx, val) for one team, seasons strictly before target_sidx.
    if len(team_hist) == 0:
        return 0.0
    w = rho ** (target_sidx - 1 - team_hist.sidx)
    dev = team_hist.val - team_hist.sidx.map(league_mean_by_season)
    n_eff = w.sum()
    weighted_dev = (w * dev).sum() / n_eff if n_eff > 0 else 0.0
    return n_eff / (n_eff + k) * weighted_dev

def one_step_ahead_features(train_seasons, valcol, rho, k):
    # Leave-one-season-out within the training block: for each training season, build its
    # shrunk effect from ONLY the seasons before it, so the axis coefficient itself is fit
    # without lookahead. rho/k are passed explicitly (not closed over) so this is safe to
    # call from both the Q4 grid search and the Q5 final fit without cross-contamination.
    rows = []
    for i in range(len(train_seasons)):
        hist_seasons = train_seasons[:i]
        tgt_season = train_seasons[i]
        tgt_sidx = season_idx[tgt_season]
        league_mean = st[st.sidx.isin([season_idx[s] for s in hist_seasons])].groupby('sidx')[valcol].mean()
        for team in st[st.season == tgt_season].team_id.unique():
            hist = st[(st.team_id == team) & (st.sidx.isin([season_idx[s] for s in hist_seasons]))][
                ['sidx', valcol]].rename(columns={valcol: 'val'})
            rows.append({'team_id': team, 'season': tgt_season,
                         'eff': shrunk_effect(hist, league_mean, tgt_sidx, rho, k)})
    return pd.DataFrame(rows)

results = []
for rho in [0.3, 0.5, 0.7, 1.0]:
    for k in [1, 2, 4, 8]:
        for valcol, target_count_col in [('xg_for_pg', 'pens_for'), ('xg_against_pg', 'pens_against')]:
            fold_devs = []
            for train_seasons, test_season in FOLDS:
                train_feat = one_step_ahead_features(train_seasons, valcol, rho, k)
                train_df = st[st.season.isin(train_seasons)].merge(train_feat, on=['team_id', 'season'], how='left')
                train_df['eff'] = train_df['eff'].fillna(0.0)
                if train_df['eff'].std() < 1e-8 or len(train_df) < 15:
                    continue
                m = smf.glm(f'{target_count_col} ~ eff', data=train_df, offset=np.log(train_df.games),
                            family=sm.families.Poisson()).fit()

                league_mean_full = st[st.sidx.isin([season_idx[s] for s in train_seasons])].groupby('sidx')[valcol].mean()
                test_sidx = season_idx[test_season]
                test = st[st.season == test_season].copy()
                effs = []
                for team in test.team_id:
                    hist = st[(st.team_id == team) & (st.sidx.isin([season_idx[s] for s in train_seasons]))][
                        ['sidx', valcol]].rename(columns={valcol: 'val'})
                    effs.append(shrunk_effect(hist, league_mean_full, test_sidx, rho, k))
                test['eff'] = effs
                pred_count = np.exp(m.params['Intercept'] + m.params['eff'] * test['eff']) * test.games
                actual_count = test[target_count_col]
                dev = 2 * np.sum(actual_count * np.log(np.where(actual_count == 0, 1, actual_count / pred_count))
                                  - (actual_count - pred_count))
                fold_devs.append(dev)
            if fold_devs:
                results.append({'rho': rho, 'k': k, 'target': target_count_col, 'mean_dev': np.mean(fold_devs)})

res_df = pd.DataFrame(results)
for target in res_df.target.unique():
    print(f'--- {target}, best rho/k ---')
    print(res_df[res_df.target == target].sort_values('mean_dev').head(3).to_string(index=False))
    print()

for target_count_col in ['pens_for', 'pens_against']:
    fold_devs = []
    for train_seasons, test_season in FOLDS:
        train_df = st[st.season.isin(train_seasons)]
        test = st[st.season == test_season]
        rate = train_df[target_count_col].sum() / train_df.games.sum()
        pred_count = rate * test.games
        actual_count = test[target_count_col]
        dev = 2 * np.sum(actual_count * np.log(np.where(actual_count == 0, 1, actual_count / pred_count))
                          - (actual_count - pred_count))
        fold_devs.append(dev)
    print(f'flat league-mean baseline, {target_count_col}: mean deviance {np.mean(fold_devs):.2f}')

--- pens_for, best rho/k ---
 rho  k   target  mean_dev
 0.3  1 pens_for 31.896653
 0.3  2 pens_for 32.014719
 0.3  4 pens_for 32.112295

--- pens_against, best rho/k ---
 rho  k       target  mean_dev
 0.7  4 pens_against 21.541466
 0.7  2 pens_against 21.554842
 0.7  8 pens_against 21.562799

flat league-mean baseline, pens_for: mean deviance 32.87
flat league-mean baseline, pens_against: mean deviance 24.19


### Reading the Q4 result

Both axes beat a flat per-season mean out of sample, and — unlike the additive PPG/GD model in
`penalty_season_rate.ipynb` — the win is on the **concession** side more than the attack side:

- **pens_for**: best (`rho=0.3, k=1`) mean deviance ≈31.9 vs flat-mean baseline ≈32.9 (~3% lower).
  `rho=0.3` decays fast — mostly *last* season's deviation matters, with only a light nudge from
  older ones. Consistent with Q3's finding that even xG isn't hugely persistent season to season.
- **pens_against**: best (`rho=0.7, k=4`) mean deviance ≈21.5 vs flat-mean baseline ≈24.2 (~11%
  lower) — a bigger, and more season-to-season-stable, edge. `rho=0.7` pools more history, which
  fits xG-against's higher persistence (r=0.31 one season out).

The concession axis is the more useful of the two — which matches the original chat hunch that
"the opponent matters," just via a duller mechanism (general danger conceded) than the flashy
same-season box-foul number Q3 ruled out.

## Q5 — Match-level scoring: does the combined structure beat the baseline?

Combine the two fitted axes exactly the way Dixon-Coles combines goal attack/defense: a team's
match-specific penalty-drawing rate is its own fitted attack rate, scaled by the opponent's
fitted defense rate *relative to average*, scaled by a home/away multiplier learned from
training data (same home/away-split idiom as `penalty_season_rate.ipynb`). Convert to a
probability via the same `P(pen) = 1 - exp(-lambda)` Poisson-trial trick, and score against:
literal production constants (`PROD_RATES['Premier_League']` from `penalty_prediction.ipynb`)
and the dynamic `baseline_home_away_league`, walk-forward, on the same seasons.

In [11]:
PROD_RATE_PL = {1: 0.157, 0: 0.101}
RHO_ATT, K_ATT = 0.3, 1
RHO_DEF, K_DEF = 0.7, 4

def fit_axis(train_seasons, test_season, rho, k, valcol, target_count_col):
    train_feat = one_step_ahead_features(train_seasons, valcol, rho, k)
    train_df = st[st.season.isin(train_seasons)].merge(train_feat, on=['team_id', 'season'], how='left')
    train_df['eff'] = train_df['eff'].fillna(0.0)
    m = smf.glm(f'{target_count_col} ~ eff', data=train_df, offset=np.log(train_df.games),
                family=sm.families.Poisson()).fit()

    league_mean_full = st[st.sidx.isin([season_idx[s] for s in train_seasons])].groupby('sidx')[valcol].mean()
    test_sidx = season_idx[test_season]
    teams = st[st.season.isin(train_seasons + [test_season])].team_id.unique()
    rates = {}
    for team in teams:
        hist = st[(st.team_id == team) & (st.sidx.isin([season_idx[s] for s in train_seasons]))][
            ['sidx', valcol]].rename(columns={valcol: 'val'})
        eff = shrunk_effect(hist, league_mean_full, test_sidx, rho, k)
        rates[team] = np.exp(m.params['Intercept'] + m.params['eff'] * eff)
    avg_rate = np.exp(m.params['Intercept'])
    return rates, avg_rate

all_preds = []
for train_seasons, test_season in FOLDS:
    attack_rate, avg_attack = fit_axis(train_seasons, test_season, RHO_ATT, K_ATT, 'xg_for_pg', 'pens_for')
    defense_rate, avg_defense = fit_axis(train_seasons, test_season, RHO_DEF, K_DEF, 'xg_against_pg', 'pens_against')

    train_matches = matches[matches.season.isin(train_seasons)]
    n_train = len(train_matches)
    baseline_home = train_matches.home_pens.sum() / n_train
    baseline_away = train_matches.away_pens.sum() / n_train
    overall_rate = (baseline_home + baseline_away) / 2
    home_mult, away_mult = baseline_home / overall_rate, baseline_away / overall_rate

    for _, row in matches[matches.season == test_season].iterrows():
        h, a = row.home_team, row.away_team
        lam_h = attack_rate.get(h, avg_attack) * (defense_rate.get(a, avg_defense) / avg_defense) * home_mult
        lam_a = attack_rate.get(a, avg_attack) * (defense_rate.get(h, avg_defense) / avg_defense) * away_mult
        all_preds.append(dict(season=test_season, target=int(row.home_pens > 0),
                               p_dc=1 - np.exp(-lam_h), p_baseline=1 - np.exp(-baseline_home), p_prod=PROD_RATE_PL[1]))
        all_preds.append(dict(season=test_season, target=int(row.away_pens > 0),
                               p_dc=1 - np.exp(-lam_a), p_baseline=1 - np.exp(-baseline_away), p_prod=PROD_RATE_PL[0]))

pred_df = pd.DataFrame(all_preds)
print(f'n rows: {len(pred_df)}\n')
for model in ['p_dc', 'p_baseline', 'p_prod']:
    ll = log_loss(pred_df.target, pred_df[model])
    br = brier_score_loss(pred_df.target, pred_df[model])
    auc = roc_auc_score(pred_df.target, pred_df[model])
    print(f'{model:12s} logloss={ll:.4f}  brier={br:.4f}  auc={auc:.4f}')

print('\nper-season log-loss:')
print(pred_df.groupby('season').apply(
    lambda d: pd.Series({m: log_loss(d.target, d[m]) for m in ['p_dc', 'p_baseline', 'p_prod']}),
    include_groups=False).round(4))

n rows: 3040

p_dc         logloss=0.3576  brier=0.1029  auc=0.5879
p_baseline   logloss=0.3606  brier=0.1035  auc=0.5451
p_prod       logloss=0.3607  brier=0.1036  auc=0.5538

per-season log-loss:
             p_dc  p_baseline  p_prod
season                               
2022-2023  0.3748      0.3748  0.3747
2023-2024  0.3755      0.3832  0.3822
2024-2025  0.3214      0.3233  0.3238
2025-2026  0.3587      0.3610  0.3621


### Reading the Q5 result

**This is the first model across all three penalty notebooks that beats the baseline in every
fold where it had any chance to differ.** Fold 1 (2022-2023) is an exact tie with baseline by
construction — with only one training season, there's no earlier season to fit the axis
coefficients from, so the model correctly degenerates to the flat rate rather than fabricating a
team effect from nothing. From fold 2 onward, once there's genuine multi-season history to pool:

| season | Dixon-Coles | baseline_home_away | prod constants |
|---|---|---|---|
| 2022-2023 (no history yet) | 0.3748 | 0.3748 | 0.3747 |
| 2023-2024 | **0.3755** | 0.3832 | 0.3822 |
| 2024-2025 | **0.3214** | 0.3233 | 0.3238 |
| 2025-2026 | **0.3587** | 0.3610 | 0.3621 |

every fold favours the attack/defense structure once there's any history to fit on, and pooled
log-loss (0.3576 vs 0.3606/0.3607) and AUC (0.588 vs 0.545/0.554) both move in its favour too. That consistency is the headline: `penalty_prediction.ipynb`'s XGBoost model
and `penalty_season_rate.ipynb`'s Poisson-on-PPG/GD model both showed a positive *average*
effect that fell apart fold-by-fold (worse than baseline in 2-3 of 4 folds each). This one
doesn't — every fold with enough history to fit on is a win, not just the average.

**Caveats, in the same spirit as the honesty in the other two notebooks:**
- The edge is modest in absolute terms (~1% log-loss reduction pooled) — real, but not the kind
  of number that changes a betting model's edge dramatically on its own.
- PL-only: `match_events`/`xt` don't cover Championship or Superligaen, so this specific
  structure can't be deployed for those leagues without either a fresh WhoScored scrape or
  falling back to xG-only axes (the attack/defense idea itself is league-agnostic — only the
  gross_xT ingredient here is PL-specific, and Q1/Q2 showed xG alone carries most of the signal
  anyway).
- Only 4 seasons of PL `match_events` history to pool over — the `rho`/`k` grid search is
  fitting a 2-parameter shrinkage model on a genuinely small number of team-seasons; treat the
  specific `rho=0.3`/`rho=0.7` values as this-sample estimates, not fixed constants, and re-check
  them once another season or two of data exists.

**For the Substack piece**, the more interesting story than "we beat the baseline by 1%" is the
shape of *why* the earlier approaches failed and this one didn't: penalty rate isn't well
explained by a team's *quality* (the Bayesian attacking-strength rating showed ~zero
correlation), and the concession side isn't a "dirty tackling" trait either (the same-season
box-foul number that looked spectacular evaporated on a persistence check) — what actually works
is duller and more mechanical: raw shot/danger volume (xG) on both sides of the ball, pooled
across seasons with the right amount of shrinkage instead of trusting any single season's number.